In [1]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [2]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    print(
        f"[START] Sanity evaluation | "
        f"train={train_name} | version={version_name}"
    )

    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:
        print(f"  -> Running phase {phase}...")

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    print(
        f"[END] Sanity evaluation DONE | "
        f"rows={len(rows)} | "
        f"phases=4"
    )

    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]


# Chạy với các version data

## V1 (Median)

In [3]:
path_1 = "/kaggle/input/lo-dataset/Median/Median"

In [4]:
path_2 = "/kaggle/input/datasets/anhtran10/lstm-results/LSTM_results"

In [5]:
df_v1 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V1 (Median)",
    train_name="train_median"
)
df_v1

[START] Sanity evaluation | train=train_median | version=V1 (Median)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median,V1 (Median),1,1,0.037007,0.992046,0.017233,0.992046,0.5,0.451577,1.0
1,train_median,V1 (Median),2,1,0.023340,0.996049,0.010011,0.996049,0.5,0.413050,1.0
2,train_median,V1 (Median),3,1,0.039118,0.991912,0.018102,0.991912,0.5,0.455274,1.0
3,train_median,V1 (Median),4,1,0.004775,0.999990,0.000007,0.999990,0.5,0.124372,1.0


In [6]:
df_v1.to_csv("results_v1.csv", index=False)

## V2 (Median CDSMOTE)

In [7]:
df_v2 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V2 (Median CDSMOTE)",
    train_name="train_median_cdsmote"
)
df_v2

[START] Sanity evaluation | train=train_median_cdsmote | version=V2 (Median CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_cdsmote,V2 (Median CDSMOTE),1,1,0.000219,0.998528,2.006952e-08,0.541797,0.5,0.041923,1.0
1,train_median_cdsmote,V2 (Median CDSMOTE),2,1,0.001071,0.999146,1.535780e-07,0.544730,0.5,0.058909,1.0
2,train_median_cdsmote,V2 (Median CDSMOTE),3,1,0.002117,0.999525,2.274419e-07,0.547840,0.5,0.062958,1.0
3,train_median_cdsmote,V2 (Median CDSMOTE),4,1,0.009679,0.999692,7.002165e-06,0.565327,0.5,0.112046,1.0


In [8]:
df_v2.to_csv("results_v2.csv", index=False)

## V3 (Median SASMOTE)

In [9]:
df_v3 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V3 (Median SASMOTE)",
    train_name="train_median_sasmote"
)
df_v3

[START] Sanity evaluation | train=train_median_sasmote | version=V3 (Median SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_sasmote,V3 (Median SASMOTE),1,1,0.002523,0.999557,1.024634e-06,0.548950,0.5,0.080937,1.0
1,train_median_sasmote,V3 (Median SASMOTE),2,1,0.002678,0.999402,4.006648e-07,0.549257,0.5,0.069217,1.0
2,train_median_sasmote,V3 (Median SASMOTE),3,1,0.006298,0.999499,2.131182e-06,0.558168,0.5,0.091697,1.0
3,train_median_sasmote,V3 (Median SASMOTE),4,1,0.009712,0.999682,6.610518e-06,0.565500,0.5,0.110981,1.0


In [10]:
df_v3.to_csv("results_v3.csv", index=False)

## V4 (Median RadiusSMOTE)

In [11]:
df_v4 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V4 (Median RadiusSMOTE)",
    train_name="train_median_radiussmote"
)
df_v4

[START] Sanity evaluation | train=train_median_radiussmote | version=V4 (Median RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_radiussmote,V4 (Median RadiusSMOTE),1,1,0.001787,0.999537,0.000646,0.546875,0.5,0.236877,1.0
1,train_median_radiussmote,V4 (Median RadiusSMOTE),2,1,0.002078,0.999624,0.000393,0.547704,0.5,0.218089,1.0
2,train_median_radiussmote,V4 (Median RadiusSMOTE),3,1,0.007401,0.999915,0.006785,0.560585,0.5,0.351979,1.0
3,train_median_radiussmote,V4 (Median RadiusSMOTE),4,1,0.006905,0.999939,0.000158,0.559553,0.5,0.188088,1.0


In [12]:
df_v4.to_csv("results_v4.csv", index=False)

## V5 (Mean)

In [13]:
path_1 = "/kaggle/input/lo-dataset/Mean/Mean"

In [14]:
df_v5 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V5 (Mean)",
    train_name="train_mean"
)
df_v5

[START] Sanity evaluation | train=train_mean | version=V5 (Mean)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean,V5 (Mean),1,1,0.000316,0.998612,4.593399e-06,0.998613,0.5,0.114811,1.0
1,train_mean,V5 (Mean),2,1,0.001362,0.999258,1.016347e-06,0.999258,0.5,0.089309,1.0
2,train_mean,V5 (Mean),3,1,0.002155,0.999513,5.173674e-08,0.999514,0.5,0.054374,1.0
3,train_mean,V5 (Mean),4,1,0.004762,0.999992,2.610671e-05,0.999992,0.5,0.153445,1.0


In [15]:
df_v5.to_csv("results_v5.csv", index=False)

## V6 (Mean CDSMOTE)

In [16]:
df_v6 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V6 (Mean CDS)",
    train_name="train_mean_cdsmote"
)
df_v6

[START] Sanity evaluation | train=train_mean_cdsmote | version=V6 (Mean CDS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_cdsmote,V6 (Mean CDS),1,1,0.015210,0.997160,8.435992e-07,0.574413,0.5,0.078919,1.0
1,train_mean_cdsmote,V6 (Mean CDS),2,1,0.017404,0.995776,4.699935e-08,0.576464,0.5,0.048790,1.0
2,train_mean_cdsmote,V6 (Mean CDS),3,1,0.014196,0.997428,6.952233e-09,0.572723,0.5,0.035453,1.0
3,train_mean_cdsmote,V6 (Mean CDS),4,1,0.009505,0.999705,1.064500e-06,0.565105,0.5,0.081850,1.0


In [17]:
df_v6.to_csv("results_v6.csv", index=False)

## V7 (Mean SASMOTE)

In [18]:
df_v7 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V7 (Mean SAS)",
    train_name="train_mean_sasmote"
)
df_v7

[START] Sanity evaluation | train=train_mean_sasmote | version=V7 (Mean SAS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_sasmote,V7 (Mean SAS),1,1,0.013106,0.996103,1.722010e-10,0.568368,0.5,0.019113,1.0
1,train_mean_sasmote,V7 (Mean SAS),2,1,0.005259,0.998339,0.000000e+00,0.554404,0.5,0.017392,1.0
2,train_mean_sasmote,V7 (Mean SAS),3,1,0.009563,0.998446,0.000000e+00,0.564258,0.5,0.017443,1.0
3,train_mean_sasmote,V7 (Mean SAS),4,1,0.010718,0.999554,8.221132e-07,0.567482,0.5,0.078453,1.0


In [19]:
df_v7.to_csv("results_v7.csv", index=False)

## V8 (Mean RadiusSMOTE)

In [20]:
df_v8 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V8 (Mean Radius)",
    train_name="train_mean_radiussmote"
)
df_v8

[START] Sanity evaluation | train=train_mean_radiussmote | version=V8 (Mean Radius)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_radiussmote,V8 (Mean Radius),1,1,0.003975,0.999948,0.002547,0.552503,0.5,0.298231,1.0
1,train_mean_radiussmote,V8 (Mean Radius),2,1,0.009660,0.999635,0.013255,0.564564,0.5,0.393990,1.0
2,train_mean_radiussmote,V8 (Mean Radius),3,1,0.005298,0.999987,0.000079,0.555883,0.5,0.167505,1.0
3,train_mean_radiussmote,V8 (Mean Radius),4,1,0.007944,0.999876,0.000095,0.561598,0.5,0.172723,1.0


In [21]:
df_v8.to_csv("results_v8.csv", index=False)

## V9 (Extra Trees)

In [22]:
path_1 = "/kaggle/input/lo-dataset/Extra_trees/Extra_trees"

In [23]:
df_v9 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V9 (Extra)",
    train_name="train_extra"
)
df_v9

[START] Sanity evaluation | train=train_extra | version=V9 (Extra)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra,V9 (Extra),1,1,0.004485,0.999960,1.647672e-03,0.999960,0.5,0.306174,1.0
1,train_extra,V9 (Extra),2,1,0.011635,0.998630,9.788521e-05,0.998630,0.5,0.191168,1.0
2,train_extra,V9 (Extra),3,1,0.021030,0.997573,1.858610e-05,0.997572,0.5,0.144880,1.0
3,train_extra,V9 (Extra),4,1,0.005078,0.999997,3.907544e-08,0.999997,0.5,0.051897,1.0


In [24]:
df_v9.to_csv("results_v9.csv", index=False)

## V10 (Extra Trees CDSMOTE)

In [25]:
df_v10 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V10 (Extra CDS)",
    train_name="train_extra_cdsmote"
)
df_v10

[START] Sanity evaluation | train=train_extra_cdsmote | version=V10 (Extra CDS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_cdsmote,V10 (Extra CDS),1,1,0.000000,0.998217,0.0,0.540852,0.5,0.017320,1.0
1,train_extra_cdsmote,V10 (Extra CDS),2,1,0.000032,0.998296,0.0,0.541013,0.5,0.017321,1.0
2,train_extra_cdsmote,V10 (Extra CDS),3,1,0.002355,0.999646,0.0,0.548511,0.5,0.017365,1.0
3,train_extra_cdsmote,V10 (Extra CDS),4,1,0.008176,0.999848,0.0,0.562291,0.5,0.017437,1.0


In [26]:
df_v10.to_csv("results_v10.csv", index=False)

## V11 (Extra Trees SASMOTE)

In [27]:
df_v11 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V11 (Extra SAS)",
    train_name="train_extra_sasmote"
)
df_v11

[START] Sanity evaluation | train=train_extra_sasmote | version=V11 (Extra SAS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_sasmote,V11 (Extra SAS),1,1,0.009576,0.997324,9.957521e-08,0.562688,0.5,0.055086,1.0
1,train_extra_sasmote,V11 (Extra SAS),2,1,0.007395,0.998319,1.015597e-08,0.559234,0.5,0.037621,1.0
2,train_extra_sasmote,V11 (Extra SAS),3,1,0.004014,0.999506,0.000000e+00,0.552702,0.5,0.017386,1.0
3,train_extra_sasmote,V11 (Extra SAS),4,1,0.008273,0.999845,0.000000e+00,0.562419,0.5,0.017438,1.0


In [28]:
df_v11.to_csv("results_v11.csv", index=False)

## V12 (Extra Trees RadiusSMOTE)

In [29]:
df_v12 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V12 (Extra Radius)",
    train_name="train_extra_radiussmote"
)
df_v12

[START] Sanity evaluation | train=train_extra_radiussmote | version=V12 (Extra Radius)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_radiussmote,V12 (Extra Radius),1,1,0.004181,0.999304,1.493852e-04,0.552926,0.5,0.185898,1.0
1,train_extra_radiussmote,V12 (Extra Radius),2,1,0.010170,0.999614,3.977920e-04,0.566526,0.5,0.219759,1.0
2,train_extra_radiussmote,V12 (Extra Radius),3,1,0.011054,0.999491,8.786287e-05,0.567656,0.5,0.170912,1.0
3,train_extra_radiussmote,V12 (Extra Radius),4,1,0.006943,0.999947,4.749594e-07,0.559555,0.5,0.071434,1.0


In [30]:
df_v12.to_csv("results_v12.csv", index=False)